## GRPO Training — trading_rl (Jensen's alpha reward)

Trains a LoRA adapter with GRPO so the model writes Backtrader strategies with positive **Jensen's alpha** vs buy-and-hold of the same stock, then evaluates base vs GRPO on DEV (2020–21) and TEST (2022–24).

1. **GPU:** *Runtime → Change runtime type* → **A100** (or L4). vLLM generation needs compute capability ≥ 8, so a T4 is only good for checking the plumbing.
2. **Secrets** (key icon, left sidebar): `hf_token`, `alpaca_key`, `alpaca_secret`.
3. **Settings** cell: `MODEL`, `NUM_STEPS`, `HUB_REPO`. Do a first run with `NUM_STEPS = 2` to check everything loads, then restart the runtime and run the full budget.
4. Run every cell top to bottom. Results are zipped and downloaded at the end.

> `trading_rl` is installed from GitHub, so local changes must be pushed first.

In [ ]:
# vllm-bnb-plugin: vLLM >= 0.28 needs it for 4-bit (bitsandbytes) models with fast_inference
# Install trading_rl (+ [train] extra: unsloth, trl, datasets, peft) + vLLM (vLLM makes GRPO generation on CUDA several times faster)
%pip install -q "trading-rl[train] @ git+https://github.com/adhamhelmy/llm-fine-tuning.git" unsloth "trl==0.22.2" datasets peft vllm vllm-bnb-plugin

In [ ]:
# Shared backtest / prompt / reward code: the `trading_rl` package in this repo
import os, sys
if os.path.isdir('../trading_rl'):   # local clone: import straight from the repo
    sys.path.insert(0, os.path.abspath('..'))
else:                                # Colab / Kaggle
    %pip install -q "trading-rl @ git+https://github.com/adhamhelmy/llm-fine-tuning.git"

### Secrets

Colab Secret manager holds these; the cell below fetches them into `os.environ`. They never touch the notebook.

In [ ]:
import os
from google.colab import userdata

# Colab secret names are case-sensitive: accept either the lowercase name or the env-var name
SECRETS = {'HF_TOKEN': 'hf_token', 'ALPACA_API_KEY': 'alpaca_key', 'ALPACA_SECRET_KEY': 'alpaca_secret'}
for env, alias in SECRETS.items():
    if os.environ.get(env):
        continue
    for name in (env, alias):
        try:
            os.environ[env] = userdata.get(name)
            break
        except Exception as e:
            err = f'{type(e).__name__}: {e}'
    else:
        print(f'Colab secret {env!r} (or {alias!r}) not readable -> {err}')

needed = {k: os.environ.get(k) for k in ('HF_TOKEN', 'ALPACA_API_KEY', 'ALPACA_SECRET_KEY')}
missing = [k for k, v in needed.items() if not v]
print('all secrets present' if not missing else 'MISSING: ' + ', '.join(missing))

### Settings

One GRPO step = 1 prompt × `num_generations` (8) completions. Rough A100 timing for a 3B model: ~1 min/step (≈3.5 h for 200 steps); a 7B model is ~2× slower. With `HUB_REPO` set, a checkpoint is pushed every `SAVE_STEPS`, so an interrupted run can resume.

In [ ]:
import torch
from trading_rl.model import Unsloth        # BEFORE trl / transformers
from trading_rl import (DEV_END, DEV_START, EVAL_SYMBOLS, TEST_END, TEST_START, TRAIN_END, TRAIN_START,
                        TRAINING_SYMBOLS, Backtester, LadderConfig, RewardConfig, make_strategy_reward, prompt_fn)
from trading_rl.dataset import build_dataset, train_windows

MODEL       = 'unsloth/Ministral-3-3B-Instruct-2512'
NUM_STEPS   = 200      # 2 = plumbing check; 150-250 for the real run
SAVE_STEPS  = 25       # checkpoint (and Hub push, if HUB_REPO is set) every N steps
HUB_REPO    = ''       # e.g. 'your-hf-username/ministral3-3b-grpo-alpha' ('' = don't push)
RESUME      = False    # True: continue from the last checkpoint in outputs/ (or pulled from HUB_REPO)
# vLLM generation (several times faster) needs compute capability >= 8 (L4/A100/H100).
FAST_INFERENCE = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
print('CUDA?', torch.cuda.is_available(), '| vLLM:', FAST_INFERENCE)

# Reward for training AND evaluation: Jensen's alpha (annualised %, vs buy-and-hold of the
# same symbol) for every strategy that trades; negative alphas squashed into (-0.99, 0) so
# trading still beats not trading (-1); capped at 50 %/yr. 'low' costs (1 + 2 bps per side).
REWARD = RewardConfig.alpha()

# GRPO: 8 completions per prompt (G=2 turned every reward gap into +-0.707), rewards scaled
# over the batch so the size of an alpha gap counts.
GRPO = dict(num_generations=8, per_device_train_batch_size=1, gradient_accumulation_steps=8,
            scale_rewards='batch', learning_rate=5e-6)

# Prompt + context for training (same as evaluation below). LadderConfig() = bare prompt.
PROMPT_CONFIG = LadderConfig(api_reference=True)

# Training prompts: every training symbol on 2016-19 and on each half (2016-17, 2018-19).
# The full window is loaded first; the halves are cut from it (no extra downloads).
backtester = Backtester(verbose=False)
WINDOWS = train_windows(TRAIN_START, TRAIN_END)
for s, e in WINDOWS:
    backtester.load_bars(TRAINING_SYMBOLS, s, e)
dataset = build_dataset(TRAINING_SYMBOLS, prompt_fn=prompt_fn(PROMPT_CONFIG, backtester),
                        windows=WINDOWS, shuffle_seed=0)
print(len(dataset), 'training prompts')

### Load the model

In [ ]:
model = Unsloth(model_name=MODEL, max_seq_length=5120, max_prompt_length=3072,
                fast_inference=FAST_INFERENCE, gpu_memory_utilization=0.6)

### Train

Saves the LoRA adapter to `grpo_saved_lora/` (and pushes it to `HUB_REPO`). Profitable strategies found during training go to `successful_strategies/`.

In [ ]:
import os
push = dict(hub_model_id=HUB_REPO, hf_token=os.environ.get('HF_TOKEN'), hub_strategy='checkpoint') if HUB_REPO else {}
trainer = model.train(
    steps=NUM_STEPS,
    reward_functions=[make_strategy_reward(backtester, reward=REWARD, save_dir='successful_strategies',
                                           save_plots=False)],
    dataset=dataset,
    save_steps=SAVE_STEPS,
    resume_from_checkpoint=RESUME,
    **push,
    **GRPO,
)

### Evaluate: base vs GRPO on DEV and TEST

Same prompt, reward and token budget as training. The base model is the same weights with the adapter switched off. `N_SAMPLES` strategies per symbol (29 eval symbols); all first-round prompts go to vLLM in one batch.

In [ ]:
import os
import pandas as pd
from trading_rl import ScoringPool, baseline_table, format_report, paired_delta, report, run_ablation_batched

EVAL_CONFIGS = [PROMPT_CONFIG]
N_SAMPLES    = 3
EVAL_TOKENS  = model.max_seq_length - model.max_prompt_length - 1   # = training completion budget
LABEL        = MODEL.split('/')[-1]
OUT          = f'ladder_results/{LABEL}'
os.makedirs(OUT, exist_ok=True)

tables = {}
for split, (start, end) in {'dev': (DEV_START, DEV_END), 'test': (TEST_START, TEST_END)}.items():
    backtester.load_bars(EVAL_SYMBOLS, start, end)
    baselines = baseline_table(backtester, EVAL_SYMBOLS, start, end, reward=REWARD)
    baselines.to_csv(f'{OUT}/baselines_{split}.csv', index=False)
    csv = f'{OUT}/eval_{split}.csv'
    with ScoringPool(workers=max(1, (os.cpu_count() or 2) - 1)) as pool:
        for label, adapter in (('base', False), ('grpo', True)):
            print(f'\n### {split}: {label}')
            run_ablation_batched(
                lambda prompts: model.generate_batch(prompts, temperature=1.0, max_new_tokens=EVAL_TOKENS,
                                                     adapter=adapter),
                backtester, EVAL_CONFIGS, EVAL_SYMBOLS, start, end, N_SAMPLES, csv,
                model=label, reward=REWARD, pool=pool)
    df = pd.read_csv(csv)
    df = df[df['reward'] == REWARD.name]
    tables[split] = (df, baselines)
    print(f'\n=== {split.upper()} ({start} -> {end}) ===')
    print(format_report(report(df, baselines, by='model'), by='model').T.to_string())
    for metric in ('profitable', 'alpha_pct', 'reward_score'):
        print(paired_delta(df, 'base', 'grpo', metric=metric, baselines=baselines, by='model'))

### Robustness report + download

Re-backtests every strategy and buy-and-hold at all cost levels (alpha, beta, Wilcoxon vs B&H, by-year, near-duplicates), then zips `ladder_results/`, `successful_strategies/` and the adapter.

In [ ]:
import shutil
from trading_rl.robustness import full_report

for split in tables:
    full_report(f'{OUT}/eval_{split}.csv', backtester, out_dir=f'{OUT}/robustness_{split}')
    print(open(f'{OUT}/robustness_{split}/report.md').read()[:3000])

shutil.make_archive('grpo_results', 'zip', '.', 'ladder_results')
for extra in ('successful_strategies', 'grpo_saved_lora'):
    if os.path.isdir(extra):
        shutil.make_archive(extra, 'zip', '.', extra)
try:
    from google.colab import files
    for f in ('grpo_results.zip', 'successful_strategies.zip', 'grpo_saved_lora.zip'):
        if os.path.exists(f):
            files.download(f)
except ImportError:
    print('zips written to', os.getcwd())